In [ ]:
# Parse the securities table
table_str = """
ticker	exchange	yahoo_symbol
SPY	ARCA	SPY
VOO	ARCA	VOO
SPYL	LSEETF	SPYL.L
SOXX	NASDAQ	SOXX
TSLA	NASDAQ	TSLA
IEUR	ARCA	IEUR
VEA	ARCA	VEA
XMME	LSEETF	XMME.L
FLKR	ARCA	FLKR
KWEB	ARCA	KWEB
ASHR	ARCA	ASHR
IEMG	ARCA	IEMG
ACWD	LSEETF	ACWD.L
SGD_CASH_VALUE	CASH	
LQD	ARCA	LQD
ZN	CBOT	ZN=F
ZF	CBOT	ZF=F
ZT	CBOT	ZT=F
GLD	ARCA	GLD
COPX	ARCA	COPX
DBMF	ARCA	DBMF
DBMF	SBF	DBMF.L
AUD	CME	6A=F
EUR	CME	6E=F
GBP	CME	6B=F
"""

In [ ]:
import pandas as pd
from io import StringIO



df_securities = pd.read_csv(StringIO(table_str), sep='\t')
yahoo_tickers = df_securities['yahoo_symbol'].dropna().tolist()

import yfinance as yf

# Download 10-year price history
price_data = yf.download(yahoo_tickers, period='10y', group_by='ticker')

# Display the data
price_data.head()

In [ ]:
import matplotlib.pyplot as plt

# Calculate rolling 3-year volatility for each ticker
window_days = 3 * 252  # 3 years * 252 trading days

for ticker in yahoo_tickers:
    if ticker in price_data.columns.levels[0]:
        # Get adjusted close or close
        if 'Adj Close' in price_data[ticker].columns:
            close = price_data[ticker]['Adj Close']
        elif 'Close' in price_data[ticker].columns:
            close = price_data[ticker]['Close']
        else:
            print(f"No close price data for {ticker}")
            continue

        # Calculate daily returns
        returns = close.pct_change().dropna()

        # Calculate rolling volatility (annualized)
        rolling_vol = returns.rolling(window=window_days).std() * (252 ** 0.5)

        # Plot
        plt.figure(figsize=(12, 6))
        rolling_vol.plot(title=f'3-Year Rolling Volatility for {ticker}')
        plt.ylabel('Volatility')
        plt.xlabel('Date')
        plt.grid(True)
        plt.show()

In [ ]:
import numpy as np

# Initialize dictionary to store metrics
vol_metrics = {}

# Parameters
trading_days_per_year = 252
window_5y = 5 * trading_days_per_year
window_2y = 2 * trading_days_per_year
ewm_halflife = 20

def adjust_future_roll_jumps(close, threshold=0.2):
    # 通过去除大跳点（如换月时的合约价差）实现连续化
    close = close.dropna().copy()
    ln = np.log(close)
    dln = ln.diff()
    jump = dln.abs() > threshold

    # 累积所有跳点对数差，并从后续数据中扣除，保持连续曲线
    adjustment = dln.where(jump, 0.0).cumsum().shift(fill_value=0.0)
    ln_adj = ln - adjustment
    return np.exp(ln_adj)

for ticker in yahoo_tickers:
    if ticker in price_data.columns.levels[0]:
        # Get adjusted close or close
        if 'Adj Close' in price_data[ticker].columns:
            close = price_data[ticker]['Adj Close']
        elif 'Close' in price_data[ticker].columns:
            close = price_data[ticker]['Close']
        else:
            continue

        # Futures continuous series (=F) may have roll jumps; 先做修正再计算收益率
        if ticker.endswith('=F'):
            # Volume-based roll detection: filter out returns on roll days
            if 'Volume' in price_data[ticker].columns:
                volume = price_data[ticker]['Volume']
                # Rolling max volume over past 10 business days * 10
                volume_filter = volume.rolling(window=10).max() * 10
                # Identify roll days where current volume > volume_filter
                roll_days = volume > volume_filter
                # Note: roll_days is a boolean series aligned with close
            else:
                roll_days = pd.Series(False, index=close.index)  # No volume data, no filtering

        # Calculate daily returns
        returns = close.pct_change().dropna()

        # For futures, filter out returns on roll days
        if ticker.endswith('=F') and 'Volume' in price_data[ticker].columns:
            # Align roll_days with returns index
            roll_days_aligned = roll_days.reindex(returns.index, fill_value=False)
            returns = returns[~roll_days_aligned]

        if len(returns) < trading_days_per_year:
            continue  # Skip if not enough data

        # 10y vol (full period)
        vol_10y = returns.std() * np.sqrt(trading_days_per_year)

        # 5y vol (last 5 years)
        returns_5y = returns.tail(window_5y)
        vol_5y = returns_5y.std() * np.sqrt(trading_days_per_year) if len(returns_5y) > 0 else np.nan

        # Rolling 3y vol
        rolling_vol = returns.rolling(window=3*trading_days_per_year).std() * np.sqrt(trading_days_per_year)
        min_3yr_vol = rolling_vol.min()
        max_3yr_vol = rolling_vol.max()

        # 2y vol exponential weighted (last 2 years, ewm halflife 20)
        returns_2y = returns.tail(window_2y)
        ewm_vol_2y = returns_2y.ewm(halflife=ewm_halflife).std().iloc[-1] * np.sqrt(trading_days_per_year) if len(returns_2y) > 0 else np.nan

        vol_metrics[ticker] = {
            '10y_vol': vol_10y,
            '5y_vol': vol_5y,
            'min_3yr_vol': min_3yr_vol,
            'max_3yr_vol': max_3yr_vol,
            'ewm_vol_2y': ewm_vol_2y
        }

# Create DataFrame
vol_df = pd.DataFrame.from_dict(vol_metrics, orient='index')
vol_df.index.name = 'ticker'
vol_df